# Phase 3 — Spark data quality audit

Run this notebook from the Dataproc Jupyter/JupyterLab interface on the cluster's Python kernel. The first cell checks that the project is billed to the one permitted Education account. It does not create a Spark session unless the account gate passes. All Spark work, outputs, Application ID, Spark Job ID(s), and run metadata remain visible in notebook cells.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import subprocess

from IPython.display import display, Markdown
from pyspark.sql import SparkSession

PROJECT_ID = os.environ.get("PROJECT_ID", "new-cloud-256902")
BILLING_ACCOUNT_ID = "01DAD8-AE5F1A-E3910A"
ALLOWED_CREDIT_ID = "3LU63BPGY77JFDQQ"
CREDIT_LIVE_VERIFIED = False
CREDIT_ID_FROM_CONSOLE = None
CREDIT_REMAINING_USD = None
RUN_COST_ESTIMATE_USD = None
COST_RESERVE_USD = 10.00
MAX_PLANNED_SPEND_USD = 40.00
BUCKET = os.environ.get("BUCKET", "")
RUN_ID = os.environ.get("RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
SOURCE_ROOT = f"gs://{BUCKET}"
MANIFEST_GCS_URI = f"gs://{BUCKET}/manifests/input_manifest.json"

if not BUCKET:
    raise ValueError("Set BUCKET to the private bucket prepared by 00_environment.ipynb.")

In [ ]:
def run_gcloud_json(args):
    result = subprocess.run(
        ["gcloud", *args, "--format=json"],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or f"gcloud exited with {result.returncode}")
    return json.loads(result.stdout or "null")

billing = run_gcloud_json(["billing", "projects", "describe", PROJECT_ID])
expected_link = f"billingAccounts/{BILLING_ACCOUNT_ID}"
billing_ready = (
    billing.get("billingAccountName") == expected_link
    and billing.get("billingEnabled") is True
)
display({
    "project_id": PROJECT_ID,
    "education_account_id": BILLING_ACCOUNT_ID,
    "billing_enabled": billing.get("billingEnabled"),
    "linked_to_education": billing.get("billingAccountName") == expected_link,
    "allowed_credit_id": ALLOWED_CREDIT_ID,
    "credit_console_verified": CREDIT_LIVE_VERIFIED,
})
if not billing_ready:
    raise RuntimeError("Stop: project is not billing-enabled on the permitted Education account.")
if not CREDIT_LIVE_VERIFIED or CREDIT_ID_FROM_CONSOLE != ALLOWED_CREDIT_ID:
    raise RuntimeError("Stop: verify the sole permitted credit ID in the Education account Credits page.")
if CREDIT_REMAINING_USD is None or RUN_COST_ESTIMATE_USD is None:
    raise RuntimeError("Stop: enter live remaining credit and a documented session estimate.")
allowed_cost = min(MAX_PLANNED_SPEND_USD, CREDIT_REMAINING_USD - COST_RESERVE_USD)
if CREDIT_REMAINING_USD <= COST_RESERVE_USD or RUN_COST_ESTIMATE_USD > allowed_cost:
    raise RuntimeError("Stop: the planned run exceeds the remaining credit after reserve.")

In [ ]:
manifest_path = Path("/tmp/input_manifest.json")
result = subprocess.run(
    ["gcloud", "storage", "cp", MANIFEST_GCS_URI, str(manifest_path)],
    check=False,
    capture_output=True,
    text=True,
)
if result.returncode:
    raise RuntimeError(result.stderr.strip() or "Could not download the input manifest.")
print(f"Loaded manifest from {MANIFEST_GCS_URI}")

manifest_bytes = manifest_path.read_bytes()
manifest = json.loads(manifest_bytes)
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()

script_path = Path("/tmp/run_quality_audit.py")
script_uri = f"gs://{BUCKET}/code/run_quality_audit.py"
result = subprocess.run(
    ["gcloud", "storage", "cp", script_uri, str(script_path)],
    check=False,
    capture_output=True,
    text=True,
)
if result.returncode:
    raise RuntimeError(result.stderr.strip() or "Could not download the audit module.")
spec = importlib.util.spec_from_file_location("run_quality_audit", script_path)
audit_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit_module)

files = audit_module.selected_files(manifest)
display({
    "manifest_sha256": manifest_sha256,
    "years": [record["year"] for record in files],
    "input_files": len(files),
})

In [ ]:
from google.cloud import storage

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET)
hash_results = []

for record in files:
    object_name = record["relative_path"]
    blob = bucket.blob(object_name)
    digest = hashlib.sha256()
    with blob.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    actual_hash = digest.hexdigest()
    expected_hash = record["sha256"]
    hash_results.append({
        "year": record["year"],
        "object": object_name,
        "sha256_matches": actual_hash == expected_hash,
    })

display(hash_results)
if not all(item["sha256_matches"] for item in hash_results):
    raise RuntimeError("Stop: at least one GCS input hash differs from the frozen manifest.")

In [ ]:
spark = (
    SparkSession.builder
    .master("yarn")
    .appName(f"rainfall-quality-audit-{RUN_ID}")
    .getOrCreate()
)
spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")

if not spark.sparkContext.master.startswith("yarn"):
    raise RuntimeError(f"Expected Dataproc YARN execution; got {spark.sparkContext.master!r}.")

application_id = spark.sparkContext.applicationId
spark_ui_url = spark.sparkContext.uiWebUrl
spark.sparkContext.setJobGroup(RUN_ID, f"Rainfall quality audit {RUN_ID}", True)

display({
    "application_id": application_id,
    "spark_master": spark.sparkContext.master,
    "spark_version": spark.version,
    "spark_ui_url": spark_ui_url,
    "run_id": RUN_ID,
})

In [ ]:
output_path = f"gs://{BUCKET}/processed/{RUN_ID}/quality-audit"
try:
    audit_module.run_audit(
        spark=spark,
        files=files,
        manifest_sha256=manifest_sha256,
        source_root=SOURCE_ROOT,
        output_path=output_path,
    )
    for dataset, row_limit in [
        ("quality_by_year", 8),
        ("reading_type_unit_distribution", 40),
        ("audit_metadata", 1),
    ]:
        print(f"Audit result: {dataset} (up to {row_limit} rows)")
        spark.read.json(f"{output_path}/{dataset}").show(row_limit, truncate=False)
finally:
    status_tracker = spark.sparkContext.statusTracker()
    job_ids = sorted(status_tracker.getJobIdsForGroup(RUN_ID))
    job_rows = []
    for job_id in job_ids:
        info = status_tracker.getJobInfo(job_id)
        job_rows.append({
            "application_id": application_id,
            "spark_job_id": job_id,
            "final_status": str(info.status) if info else "not available",
        })
    display(job_rows)
    spark.sparkContext.clearJobGroup()

print(f"Audit output: {output_path}")
display(Markdown(
    f"**Spark Web UI:** [{spark_ui_url}]({spark_ui_url})  \n"
    f"**Application ID:** `{application_id}`  \n"
    f"**Spark Job ID(s):** {job_ids}"
))

## Screenshot each Spark job

Wait until all jobs created by this cell reach a final state. In the Spark Web UI, capture the Jobs summary and the detail page for each Job ID listed above. Save the original screenshots under `report/evidence/spark-ui/<RUN_ID>/` and update `report/evidence/spark-ui/index.csv` with this notebook action, Application ID, all Job IDs, final status, duration, and screenshot paths. For a finished application, use Spark History Server. A screenshot from Dataproc Jobs or YARN does not satisfy this requirement.